# 03 — EDA, sentiment và xu hướng Reddit

Đơn vị quan sát là **bài đăng**, không phải người dùng riêng lẻ: dataset không có bình luận và pipeline đã bỏ tên tác giả. Vì vậy diễn giải kết quả là cảm xúc thể hiện trong bài đăng, tổng hợp theo thời gian/subreddit; không suy rộng thành cảm xúc cá nhân của người dùng.

Audit/ETL cho thấy 545,427 bài từ 19 subreddit, giai đoạn 2008-03 đến 2022-05; một số tháng/năm bị khuyết và số bài giữa cộng đồng không cân bằng. Notebook này tạo các biểu đồ coverage, VADER sentiment và xu hướng theo tháng.


In [ ]:
%pip -q install pandas pyarrow matplotlib seaborn vaderSentiment


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DATA_DIR = Path("/content/data/processed")
CLEAN_PATH = DATA_DIR / "reddit_posts_clean.parquet"
SCORED_PATH = DATA_DIR / "reddit_posts_sentiment.parquet"
assert CLEAN_PATH.exists(), f"Chưa thấy {CLEAN_PATH}; chạy 02_etl.ipynb trước."


In [ ]:
# 1) Coverage theo tháng và subreddit
df = pd.read_parquet(CLEAN_PATH, columns=["id", "subreddit", "created_at_utc", "text", "score", "num_comments"])
df["month"] = df["created_at_utc"].dt.to_period("M").astype(str)
monthly_n = df.groupby("month").size().sort_index()
calendar = pd.period_range(df["created_at_utc"].min().to_period("M"), df["created_at_utc"].max().to_period("M"), freq="M").astype(str)
missing_months = sorted(set(calendar) - set(monthly_n.index))
print("Dòng:", len(df), "| subreddit:", df["subreddit"].nunique())
print("Tháng có bài:", len(monthly_n), "/", len(calendar))
print("Tháng không có dòng:", missing_months)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), constrained_layout=True)
monthly_n.plot(ax=axes[0], color="#3366aa", linewidth=1)
axes[0].set(title="Số bài Reddit theo tháng", xlabel="Tháng (UTC)", ylabel="Số bài")
axes[0].grid(alpha=.2)
monthly_n.rolling(3, min_periods=1).mean().plot(ax=axes[1], color="#d65f5f")
axes[1].set(title="Trung bình trượt 3 tháng (tháng trống vẫn cần được diễn giải thận trọng)", xlabel="Tháng (UTC)", ylabel="Bài/tháng")
axes[1].grid(alpha=.2)
plt.show()

sub_n = df["subreddit"].value_counts()
sub_share = (sub_n / sub_n.sum() * 100).rename("share_pct").to_frame()
sub_share["posts"] = sub_n
display(sub_share.sort_values("posts", ascending=False))


In [ ]:
# 2) VADER scoring theo batches; giữ kết quả để không phải tính lại
if not SCORED_PATH.exists():
    analyzer = SentimentIntensityAnalyzer()
    pf = pq.ParquetFile(CLEAN_PATH)
    writer = None
    for batch in pf.iter_batches(batch_size=25_000, columns=["id", "subreddit", "created_at_utc", "text", "score", "num_comments"]):
        part = batch.to_pandas()
        scores = [analyzer.polarity_scores(str(text))["compound"] for text in part["text"]]
        part["sentiment_compound"] = scores
        part["sentiment_label"] = pd.cut(
            part["sentiment_compound"],
            bins=[-1.000001, -0.05, 0.05, 1.000001],
            labels=["negative", "neutral", "positive"],
            include_lowest=True,
        ).astype("string")
        table = pa.Table.from_pandas(part, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(SCORED_PATH, table.schema, compression="zstd")
        writer.write_table(table)
    if writer is not None:
        writer.close()
    print("Saved:", SCORED_PATH)
else:
    print("Dùng sentiment đã tính sẵn:", SCORED_PATH)


In [ ]:
# 3) Phân bố sentiment và xu hướng theo thời gian
sent = pd.read_parquet(SCORED_PATH)
sent["month"] = sent["created_at_utc"].dt.to_period("M").astype(str)
label_share = sent["sentiment_label"].value_counts(normalize=True).mul(100).reindex(["negative", "neutral", "positive"])
print("Tỷ lệ nhãn VADER (%):")
display(label_share.rename("percent").to_frame())

monthly = sent.groupby("month", as_index=False).agg(
    posts=("id", "size"),
    mean_compound=("sentiment_compound", "mean"),
    median_compound=("sentiment_compound", "median"),
    positive_share=("sentiment_label", lambda x: (x == "positive").mean()),
    negative_share=("sentiment_label", lambda x: (x == "negative").mean()),
)
monthly.to_csv(DATA_DIR / "sentiment_monthly.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), constrained_layout=True)
label_share.plot(kind="bar", ax=axes[0], color=["#d65f5f", "#999999", "#55a868"])
axes[0].set(title="Phân bố sentiment theo VADER", xlabel="Nhãn", ylabel="Tỷ lệ (%)")
monthly.set_index("month")["mean_compound"].plot(ax=axes[1], color="#3366aa")
axes[1].axhline(0, color="black", linewidth=.8)
axes[1].set(title="Sentiment trung bình theo tháng", xlabel="Tháng (UTC)", ylabel="VADER compound trung bình")
axes[1].grid(alpha=.2)
plt.show()


In [ ]:
# 4) So sánh subreddit và xuất mẫu để con người rà nhãn sentiment
by_subreddit = sent.groupby("subreddit", as_index=False).agg(
    posts=("id", "size"),
    mean_compound=("sentiment_compound", "mean"),
    positive_share=("sentiment_label", lambda x: (x == "positive").mean()),
    negative_share=("sentiment_label", lambda x: (x == "negative").mean()),
).sort_values("posts", ascending=False)
by_subreddit.to_csv(DATA_DIR / "sentiment_by_subreddit.csv", index=False)
display(by_subreddit)

# Mẫu cân bằng theo nhãn để rà chất lượng; không dùng tỷ lệ mẫu này để ước lượng prevalence.
audit_parts = []
for label, group in sent.groupby("sentiment_label", observed=True):
    audit_parts.append(group.sample(n=min(50, len(group)), random_state=42))
audit_sample = pd.concat(audit_parts, ignore_index=True)[["id", "subreddit", "created_at_utc", "text", "sentiment_compound", "sentiment_label"]]
audit_sample["human_label"] = ""
audit_sample["review_notes"] = ""
audit_sample.to_csv(DATA_DIR / "sentiment_manual_audit.csv", index=False)
print("Manual audit sample:", DATA_DIR / "sentiment_manual_audit.csv", "rows:", len(audit_sample))


## Diễn giải và giới hạn

- VADER là baseline từ từ điển/quy tắc cho văn bản mạng xã hội tiếng Anh; kết quả là score tự động, chưa phải nhãn chuẩn. Rà `sentiment_manual_audit.csv` bằng người đọc và ghi lỗi mỉa mai, từ chuyên ngành, câu hỏi, link-only. Mẫu được cân bằng theo nhãn nên không dùng để tính tỷ lệ cảm xúc của toàn bộ tập.
- Các tháng thiếu dữ liệu phải hiển thị là thiếu, không xem như tháng có 0 bài. Số bài thay đổi theo thời gian cũng có thể phản ánh độ bao phủ nguồn thu thập.
- Subreddit có quy mô rất khác nhau; so sánh sentiment bằng tỷ lệ/điểm trung bình kèm số mẫu, không chỉ so sánh tổng số bài.
- Sentiment trong title/body là cảm xúc của nội dung bài đăng. Dataset không có comment tree/parent ID nên chưa thể làm network analysis như một phần tái hiện Patel et al.
